In [ ]:
# ============================================
# One-Class Room Filter (Prototype + Cosine)
# (fin_artifacts / fin_reports 저장 + LOO CSV/PNG)
# ============================================

import os, glob, json, time, random, csv
from pathlib import Path

import torch
import numpy as np
from PIL import Image
from torchvision import transforms, models
from torch.nn.functional import normalize

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# 설정
EMPTYROOM_DIR = "./empty_room"
TEST_DIR      = "test"
ARTIFACT_DIR  = "fin_artifacts"
REPORT_DIR    = "fin_reports"
BATCH_SIZE = 64
CONF_Q = 0.05  # LOO 하위 5% 컷

# 유틸
def ensure_dirs():
    Path(ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)
    Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)

def list_images(folder):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
    paths = []
    for e in exts:
        paths += glob.glob(os.path.join(folder, e))
    return sorted(paths)

def set_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 모델 준비
def build_backbone(device):
    # 가중치 로딩
    try:
        weights_enum = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights_enum)
        mean, std = weights_enum.meta["mean"], weights_enum.meta["std"]
    except Exception:
        try:
            model = models.efficientnet_b0(pretrained=True)
        except Exception:
            model = models.efficientnet_b0()
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]

    model.classifier = torch.nn.Identity()
    model.eval().to(device)

    T = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return model, T

@torch.no_grad()
def embed_images(paths, model, T, device, batch=BATCH_SIZE):
    if not paths:
        return torch.empty(0, 1280)
    embs = []
    for i in range(0, len(paths), batch):
        imgs = [T(Image.open(p).convert("RGB")) for p in paths[i:i+batch]]
        x = torch.stack(imgs).to(device)
        e = model(x)
        e = normalize(e, dim=1)
        embs.append(e.cpu())
    return torch.cat(embs, dim=0)


# LOO 임계값 계산 (+ 점수 반환)
def loo_cosine_threshold(E, q=CONF_Q):
    """
    E: (N,D) L2 정규화 임베딩(CPU 텐서)
    반환: tau(float), stats(dict), scores(np.ndarray shape=(N,))
    """
    sums = E.sum(0, keepdim=True)      # (1,D)
    S = normalize(sums - E, dim=1)     # (N,D)
    scores = (E * S).sum(1).numpy()    # (N,)
    tau = float(np.quantile(scores, q))
    stats = {
        "mean": float(scores.mean()),
        "std": float(scores.std(ddof=1)),
        "min": float(scores.min()),
        "tau": tau,
        "q": q,
    }
    return tau, stats, scores


# 모델 / 아티팩트 저장
def save_model(model, save_path=None):
    if save_path is None:
        save_path = os.path.join(ARTIFACT_DIR, "room_model.pth")
    torch.save(model.state_dict(), save_path)
    print(f"모델 저장 완료 → {save_path}")

def save_artifacts(mu, tau, meta):
    np.save(os.path.join(ARTIFACT_DIR, "prototype_mu.npy"), mu.cpu().numpy())
    with open(os.path.join(ARTIFACT_DIR, "threshold.json"), "w", encoding="utf-8") as f:
        json.dump({"tau": tau}, f, indent=2, ensure_ascii=False)
    with open(os.path.join(ARTIFACT_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print("프로토타입/임계값/메타 저장 완료")


# 훈련 LOO 점수 리포트/히스토그램 저장
def save_training_loo_report(train_paths, loo_scores, tau):
    ts = time.strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(REPORT_DIR, f"train_loo_{ts}.csv")
    png_path = os.path.join(REPORT_DIR, f"train_loo_hist_{ts}.png")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["path", "cosine", "sim01", "is_outlier(<tau)"])
        for p, s in zip(train_paths, loo_scores):
            w.writerow([p, round(float(s), 6), round((s+1)/2, 6), int(s < tau)])
    print(f"훈련 LOO 점수 CSV 저장 → {csv_path}")

    # 히스토그램 PNG 저장 (배경 흰색 + 임계값 보조선)
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"
    plt.figure(figsize=(8, 4.5))
    plt.hist(loo_scores, bins=50)
    plt.axvline(tau, linestyle="--")
    plt.title("Training LOO Cosine Score Distribution")
    plt.xlabel("LOO cosine score")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(png_path, dpi=160)
    plt.close()
    print(f"훈련 LOO 히스토그램 저장 → {png_path}")


# 추론
@torch.no_grad()
def cosine_score(img_path, model, T, device, mu):
    x = T(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    e = normalize(model(x), dim=1)
    mu = mu.to(device)
    return float((e * mu).sum())

def sim01(c):
    return (c + 1) / 2

def predict_label(score, tau):
    return "room" if score >= tau else "other"

def run_inference_folder(test_dir, model, T, device, mu, tau):
    paths = list_images(test_dir)
    if not paths:
        print(f("테스트 폴더 비어 있음: {test_dir}"))
        return None
    rows = []
    for p in paths:
        c = cosine_score(p, model, T, device, mu)
        rows.append({
            "path": p,
            "cosine": round(c, 6),
            "sim01": round(sim01(c), 6),
            "label": predict_label(c, tau),
            "tau": round(tau, 6),
        })
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_csv = os.path.join(REPORT_DIR, f"pred_{ts}.csv")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"[REPORT] 예측 결과 저장: {out_csv} (총 {len(rows)}건)")
    return out_csv

# -----------------------------
# 학습 및 저장

def train_and_calibrate(emptyroom_dir, device):
    room_paths = list_images(emptyroom_dir)
    if not room_paths:
        raise RuntimeError("방 이미지가 없습니다.")
    model, T = build_backbone(device)
    E = embed_images(room_paths, model, T, device)
    mu = normalize(E.mean(0, keepdim=True), dim=1).to(device)

    tau, stats, loo_scores = loo_cosine_threshold(E, q=CONF_Q)

    meta = {
        "n_train": len(room_paths),
        "conf_q": CONF_Q,
        "created": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    save_artifacts(mu, tau, meta)
    save_model(model)

    print(f"[TRAIN] N={len(room_paths)}, mean={stats['mean']:.4f}, tau={stats['tau']:.4f}")
    # LOO 점수 CSV/PNG까지 저장
    save_training_loo_report(room_paths, loo_scores, tau)
    return model, T, mu, tau


def main():
    ensure_dirs()
    device = set_device()
    print(f"[DEVICE] {device}")

    model, T, mu, tau = train_and_calibrate(EMPTYROOM_DIR, device)

    if os.path.isdir(TEST_DIR):
        run_inference_folder(TEST_DIR, model, T, device, mu, tau)

if __name__ == "__main__":
    main()

[DEVICE] cpu
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Playdata/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:11<00:00, 1.84MB/s]
c:\Users\Playdata\miniconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Playdata\miniconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[SAVE] 프로토타입/임계값/메타 저장 완료
[SAVE] 모델 저장 완료 → fin_artifacts\room_model.pth
[TRAIN] N=100, mean=0.6883, tau=0.4716
[REPORT] 훈련 LOO 점수 CSV 저장 → fin_reports\train_loo_20251022_140627.csv
[REPORT] 훈련 LOO 히스토그램 저장 → fin_reports\train_loo_hist_20251022_140627.png
